In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import numpy as np
import sys
from pathlib import Path
import os

repo_root = Path.cwd().parent.resolve()          # repo root
sys.path.insert(0, str(repo_root))

from modules.chemeleon_embds import chemeleon_embedding
from modules.molclr_embds import mol_clr_embed
from modules.maceoff_embds import generate_maceoff_embedding
from modules.mlp_tuning import tune_mlp_with_fixed_split
from modules.HQ_featureextractor import chemdescriptor

from rdkit import Chem
import numpy as np
import pandas as pd

from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# Load Data

In [ ]:
def get_morgan_fps(df):
    """Get Morgan fingerprints for a dataframe of SMILES strings."""
    mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]
    # Morgan fingerprints -> numpy array 
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048, useChirality=False) for m in mols]
    return fps

def get_morgan_fp_array(df):
    """Get Morgan fingerprints as a numpy array."""
    fps = get_morgan_fps(df)
    arr = np.zeros((len(fps), 2048), dtype=int)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr

In [ ]:
def load_csv_by_subdir(datasets_dir: str | Path) -> dict[str, object]:
    datasets_dir = Path(datasets_dir).resolve()
    data_dict: dict[str, object] = {}

    for subdir in sorted(p for p in datasets_dir.iterdir() if p.is_dir()):
        csv_files = sorted(subdir.glob("*.csv"))

        data_dict[subdir.name] = pd.read_csv(csv_files[0])

    return data_dict

# example
targets = load_csv_by_subdir("../01_DATASETS/")

## Generate Baseline Embeddings

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_morgan_fp(smiles, fp_gen, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = fp_gen.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

morgan_fps_dict = {}
for name, df in targets.items():
    print(f"Processing {name}...")
    fp_gen = Chem.rdFingerprintGenerator.GetMorganGenerator(2)
    fps = df['SMILES'].apply(lambda s: smiles_to_morgan_fp(s, fp_gen))
    # Stack the fingerprint arrays into a 2D array
    fps_matrix = np.vstack(fps.values)
    morgan_fps_dict[name] = fps_matrix
    np.save(os.path.join("../02_EMBEDDINGS", name, "morgan_fps_embeddings.npy"), fps_matrix)

In [ ]:
# make QU descriptors
for target, data in targets.items():
    base_dir = f"../02_EMBEDDINGS/{target}/"
    print(f"Processing target: {target}")
    rows = [chemdescriptor(smiles) for smiles in data["SMILES"].values]
    arr = pd.DataFrame(rows).to_numpy()
    np.save(os.path.join(base_dir, "hq_descriptor_embeddings.npy"), arr)

# Generate Embds with Pre-Trained Encoders 

In [ ]:
def safe_save_npy(path, arr):
    try:
        np.save(path, np.asarray(arr))
    except ValueError as e:
        print(f"[WARN] np.save failed for {path}: {e}")
        fallback = path.replace(".npy", ".object.npy")
        np.save(fallback, np.array(arr, dtype=object), allow_pickle=True)
        print(f"[WARN] Saved ragged data as object array: {fallback}")

def generate_and_save_embeddings(name, smiles_list, save_dir="../02_EMBEDDINGS"):
    save_dir = os.path.join(save_dir, name)
    os.makedirs(save_dir, exist_ok=True)

    #CheMeleon
    chemeleon_emb = chemeleon_embedding(smiles_list, None, ckpt_path="../04_FM_CPS/chemeleon_mp.pt")
    safe_save_npy(os.path.join(save_dir, "chemeleon_embeddings.npy"), chemeleon_emb)


    # MolCLR
    molclr_emb = mol_clr_embed(smiles_list, ckpt_path="04_FM_CPS/MolCLR/ckpt/pretrained_gin/checkpoints/model.pth")
    molclr_emb = molclr_emb.numpy()
    safe_save_npy(os.path.join(save_dir, "molclr_embeddings.npy"), molclr_emb)

    # Mace off large
    maceoff_emb, failed_mols = generate_maceoff_embedding(smiles_list, model_size="large", num_confs=1)
    safe_save_npy(os.path.join(save_dir, "maceoff_large_embeddings.npy"), maceoff_emb)
    print(f"Failed molecules for {name} (large): {failed_mols}")

    return {
        "chemeleon": chemeleon_emb,
        "molclr": molclr_emb,
        "maceoff": maceoff_emb
    }

embds = {}
for dataset_name, data in targets.items():
        print(f"Generating embeddings for {dataset_name}...")
        embds[dataset_name] = generate_and_save_embeddings(dataset_name, data["SMILES"])